# Avro Basics — 03: Schema Definition


Avro schemas are defined in JSON. Understanding Avro types is essential
for building correct schemas for your data.

Topics: primitive types, records, arrays, maps, unions, enums, fixed,
converting between Avro JSON schema and Spark StructType.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 12:27:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Avro Primitive Types



In [2]:

print("""
Avro primitive types:
  null    → NullType
  boolean → BooleanType
  int     → IntegerType
  long    → LongType
  float   → FloatType
  double  → DoubleType
  bytes   → BinaryType
  string  → StringType

Logical types (annotations on primitives):
  {"type":"long","logicalType":"timestamp-micros"} → TimestampType
  {"type":"int", "logicalType":"date"}             → DateType
  {"type":"long","logicalType":"time-micros"}      → no direct Spark mapping
""")



Avro primitive types:
  null    → NullType
  boolean → BooleanType
  int     → IntegerType
  long    → LongType
  float   → FloatType
  double  → DoubleType
  bytes   → BinaryType
  string  → StringType

Logical types (annotations on primitives):
  {"type":"long","logicalType":"timestamp-micros"} → TimestampType
  {"type":"int", "logicalType":"date"}             → DateType
  {"type":"long","logicalType":"time-micros"}      → no direct Spark mapping



## Step 2 — Record Type (Nested Struct)



In [3]:
import json as pyjson
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

RECORD_SCHEMA = pyjson.dumps({
    "type": "record",
    "name": "Order",
    "namespace": "com.example",
    "fields": [
        {"name": "order_id",  "type": "long"},
        {"name": "customer",  "type": {
            "type": "record", "name": "Customer",
            "fields": [
                {"name": "id",      "type": "long"},
                {"name": "name",    "type": "string"},
                {"name": "country", "type": "string"},
            ]
        }},
        {"name": "amount",    "type": "double"},
        {"name": "status",    "type": "string"},
    ]
})

# Must use explicit StructType — createDataFrame with tuples creates _1,_2,_3
# which doesn't match the Avro schema field names (id, name, country)
nested_schema = StructType([
    StructField("order_id", LongType()),
    StructField("customer", StructType([
        StructField("id",      LongType()),
        StructField("name",    StringType()),
        StructField("country", StringType()),
    ])),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
])

from pyspark.sql import Row
nested_data = spark.createDataFrame([
    Row(order_id=1, customer=Row(id=101, name="Alice", country="US"), amount=999.99, status="confirmed"),
    Row(order_id=2, customer=Row(id=102, name="Bob",   country="UK"), amount=499.99, status="shipped"),
    Row(order_id=3, customer=Row(id=103, name="Carol", country="DE"), amount=299.99, status="pending"),
], schema=nested_schema)

OUT = f"{DATA_DIR}/nested_record"
nested_data.write.format("avro").option("avroSchema", RECORD_SCHEMA).mode("overwrite").save(OUT)
print("Written nested Avro record:")
df_back = spark.read.format("avro").load(OUT)
df_back.show(truncate=False)
df_back.printSchema()

Written nested Avro record:
+--------+----------------+------+---------+
|order_id|customer        |amount|status   |
+--------+----------------+------+---------+
|1       |{101, Alice, US}|999.99|confirmed|
|3       |{103, Carol, DE}|299.99|pending  |
|2       |{102, Bob, UK}  |499.99|shipped  |
+--------+----------------+------+---------+

root
 |-- order_id: long (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)



## Step 3 — Array and Map Types



In [4]:

arr_schema = pyjson.dumps({
    "type": "record", "name": "Product",
    "fields": [
        {"name": "id",     "type": "long"},
        {"name": "tags",   "type": {"type": "array",  "items": "string"}},
        {"name": "prices", "type": {"type": "map",    "values": "double"}},
    ]
})

arr_data = spark.createDataFrame([
    (1, ["tech","premium"], {"USD": 999.99, "EUR": 919.99}),
    (2, ["budget"],         {"USD": 29.99,  "EUR": 27.99}),
], ["id","tags","prices"])

OUT_ARR = f"{DATA_DIR}/arrays_maps"
arr_data.write.format("avro").option("avroSchema", arr_schema).mode("overwrite").save(OUT_ARR)
print("Avro with arrays and maps:")
spark.read.format("avro").load(OUT_ARR).show(truncate=False)


Avro with arrays and maps:
+---+---------------+------------------------------+
|id |tags           |prices                        |
+---+---------------+------------------------------+
|1  |[tech, premium]|{EUR -> 919.99, USD -> 999.99}|
|2  |[budget]       |{EUR -> 27.99, USD -> 29.99}  |
+---+---------------+------------------------------+



## Step 4 — Union (Nullable Fields)

Union with null is how Avro represents nullable fields.

In [5]:
union_schema = pyjson.dumps({
    "type": "record", "name": "Event",
    "fields": [
        {"name": "id",      "type": "long"},
        {"name": "value",   "type": ["null","double"],  "default": None},
        {"name": "label",   "type": ["null","string"],  "default": None},
        {"name": "tag",     "type": ["string","null"],  "default": "unknown"},
    ]
})

# Must use explicit schema — tuples with None need nullable StructField
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
from pyspark.sql import Row

union_schema_spark = StructType([
    StructField("id",    LongType(),   nullable=False),
    StructField("value", DoubleType(), nullable=True),
    StructField("label", StringType(), nullable=True),
    StructField("tag",   StringType(), nullable=True),
])

union_data = spark.createDataFrame([
    Row(id=1, value=99.99, label="gold",   tag="vip"),
    Row(id=2, value=None,  label=None,     tag="standard"),
    Row(id=3, value=49.99, label="silver", tag=None),
], schema=union_schema_spark)

OUT_U = f"{DATA_DIR}/unions"
union_data.write.format("avro").option("avroSchema", union_schema).mode("overwrite").save(OUT_U)
df_u = spark.read.format("avro").load(OUT_U)
print("Avro union (nullable fields):")
df_u.show()
df_u.printSchema()
print('["null","string"] in Avro -> StringType nullable=true in Spark')

Avro union (nullable fields):
+---+-----+------+--------+
| id|value| label|     tag|
+---+-----+------+--------+
|  1|99.99|  gold|     vip|
|  3|49.99|silver|    NULL|
|  2| NULL|  NULL|standard|
+---+-----+------+--------+

root
 |-- id: long (nullable = true)
 |-- value: double (nullable = true)
 |-- label: string (nullable = true)
 |-- tag: string (nullable = true)

["null","string"] in Avro -> StringType nullable=true in Spark


## Step 5 — StructType to Avro Schema Conversion



In [6]:

from pyspark.sql.avro.functions import to_avro, from_avro

# Spark can serialize a column as Avro binary using to_avro
sample = spark.createDataFrame([(1,"Alice",99.99),(2,"Bob",49.99)], ["id","name","amount"])
avro_binary = sample.select(to_avro(F.struct("*")).alias("avro_bytes"))
print("Serialized as Avro binary column:")
avro_binary.show(truncate=False)
print("This is the Kafka Avro message format (without schema registry header).")


Serialized as Avro binary column:
+-------------------------------------------------------+
|avro_bytes                                             |
+-------------------------------------------------------+
|[00 02 00 0A 41 6C 69 63 65 00 8F C2 F5 28 5C FF 58 40]|
|[00 04 00 06 42 6F 62 00 1F 85 EB 51 B8 FE 48 40]      |
+-------------------------------------------------------+

This is the Kafka Avro message format (without schema registry header).


## Summary



In [7]:

print("""
Avro type → Spark type mapping:
  null          → NullType
  boolean       → BooleanType
  int           → IntegerType
  long          → LongType
  float         → FloatType
  double        → DoubleType
  string        → StringType
  bytes         → BinaryType
  array[T]      → ArrayType(T)
  map[string,T] → MapType(StringType, T)
  record        → StructType
  ["null","T"]  → T (nullable)  ← most common nullable pattern
  enum          → StringType
  fixed         → BinaryType

  logicalType:date            → DateType
  logicalType:timestamp-micros → TimestampType
""")



Avro type → Spark type mapping:
  null          → NullType
  boolean       → BooleanType
  int           → IntegerType
  long          → LongType
  float         → FloatType
  double        → DoubleType
  string        → StringType
  bytes         → BinaryType
  array[T]      → ArrayType(T)
  map[string,T] → MapType(StringType, T)
  record        → StructType
  ["null","T"]  → T (nullable)  ← most common nullable pattern
  enum          → StringType
  fixed         → BinaryType

  logicalType:date            → DateType
  logicalType:timestamp-micros → TimestampType

